# Auditoria historica de controles e referencias

**Papel nesta entrega:** Suplementar: analise historica descrita no trabalho

**Origem:** https://colab.research.google.com/drive/1Ik3FyU7z4uO9MXVLyAPKSTGBlkzIs3cD

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Dependencias e caminhos sao os da campanha historica, nao das quatro execucoes padronizadas. Esses cadernos nao devem substituir os resultados principais. Consulte o mapa de cobertura e mantenha separadas as datas de execucao. 

# Etapa 05 - Auditoria dos controles DMSO e das referencias P95

**Pergunta:** as diferencas de contraste entre placas tambem aparecem nos controles DMSO de R1-R3 ou nas referencias usadas para calcular os escores?

Esta auditoria examina os **escores originais por poco**, as **referencias originais** e os **modelos numericos congelados da Etapa 04**. Nao recalibra escores, nao ajusta novos modelos e nao reclassifica candidatos. Nenhum arquivo de R4 e necessario.

Execute **Ambiente de execucao > Executar tudo**. As fontes fixas e seus SHA-256 sao conferidos antes dos calculos. Cada execucao cria uma pasta nova, sem sobrescrever entradas ou resultados anteriores.

Os calculos numericos e as dez figuras foram testados localmente com os dados reais. A leitura de Parquet deste notebook usa Polars no Colab. A verificacao local nao equivale a uma execucao na sua sessao do Colab; este notebook esta entregue sem contagens de execucao fabricadas.

## 1. Ambiente e acesso ao Drive
Os pacotes so serao instalados se estiverem ausentes. Nao ha reinstalacao da normalizacao ou treinamento de modelos.

In [ ]:
from pathlib import Path
import os, sys, importlib.util, subprocess
for module, package in [('numpy','numpy'),('pandas','pandas'),('polars','polars'),('matplotlib','matplotlib'),('scipy','scipy')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
from IPython.display import display, Markdown, Image
RAIZ = Path(os.environ.get('TCC_RAIZ', '/content/drive/MyDrive/mba_tcc'))
print('Raiz:', RAIZ)


## 2. O que sera auditado

**RMS (E)** e a magnitude das medidas ja normalizadas por MAD. Nao e intensidade bruta de fluorescencia.

O escore original e $A_{pb}=E_{pb}/q_{pb}$, em que $q_{pb}$ e o P95 de 84 DMSO da placa $p$ e do bloco $b$, reunindo R1-R3.

O contraste usado no agrupamento, antes da escala robusta adicional, e
$$C_{A,b}=\log(1+A_b)-\frac{1}{4}\sum_{j=1}^{4}\log(1+A_j).$$

A auditoria compara esse contraste com a centralizacao analoga de $\log(1+E)$, chamada $C_E$, e calcula $\Delta_{P95}=C_A-C_E$. A igualdade $C_A=C_E+\Delta_{P95}$ e **algebrica, nao causal**. A mudanca depende do RMS e da referencia. Nenhuma porcentagem de artefato sera estimada.

Diferencas no P95 podem refletir caudas distintas mesmo com medianas semelhantes. O codigo identifica os dois controles que delimitam a interpolacao do P95, sem excluir nenhum controle. A taxa de excedencia dos 84 controles usados na propria calibracao nao e teste independente.

As projecoes de DMSO usam os centros e escalas salvos: pertencer a G02 nao transforma um controle em hit ou falso positivo.

### Fontes, integridade e reproducao dos resultados originais

In [ ]:
"""Etapa 05. Auditoria descritiva dos controles de R1-R3 e referencias P95.
Nao recalibra os escores, nao reajusta modelos, nao altera candidatos e nao le R4.
"""
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
import hashlib, json, platform, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P,R,E,D,W='Metadata_Plate_ID','Metadata_Replicate','Metadata_EOS','Metadata_Concentration','Metadata_Well'
KEY=[P,E,'_dose_key']
REPS=['R1','R2','R3']
PLATES=[f'B100{i}' for i in range(1,8)]
CHANNELS=['AGP','DNA','ER','Mito']
BLOCKS=CHANNELS+['Morfologia_Cells','Morfologia_Cyto','Morfologia_Nuc']
EXPECTED_COUNTS=dict(AGP=98,DNA=158,ER=130,Mito=152,Morfologia_Cells=42,Morfologia_Cyto=38,Morfologia_Nuc=41)
FS_REL=Path('pipeline_IMTM_HepG2/20260919T204105_625149Z/normalizacao_20260919T205516_579360Z/feature_selection_20260919T211235_656816Z')
ES_REL=FS_REL/'escores_exploratorios_20260919T230543_380473Z'
CL_REL=Path('resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z')
MAP_REL=Path('resultados_tcc/mapa_multibloco/R1_R3_20260926T152153_537184Z')
SOURCES={
 'scores':str(ES_REL/'escores_pocos_R1_R3.parquet'),
 'refs':str(ES_REL/'referencias_dmso.csv'),
 'complete':str(MAP_REL/'tabelas/completas.csv'),
 'models':str(CL_REL/'modelos_numericos.json'),
 'groups':str(CL_REL/'tabelas/grupos.csv'),
 'assign':str(CL_REL/'tabelas/atribuicoes.csv')}
SOURCE_IDS={
 'scores':'1p9BtOuWnUo6Y-AY9bGXaPzRUuCUvn5lS','refs':'1SgytsR_sz-st5I7Q0ive0SewOZmh5yKx',
 'complete':'1vB1LoU1oZQg1CxjYqKbRClxhfsNblU2P','models':'1GDmrHEMOFWuZPj4Zt57qVU7qqNTKGK63',
 'groups':'1wf33vAKy6IBYswYG1t7e8HMFkmpdI1eA','assign':'19-cPZoxd0JTW6jOLP_IZ4Md9cVsWJZys'}
HASHES={
 'scores':'3553535af6dc74f9a77151c2d2323846b92fbff7114e430842607fb136673e4e',
 'refs':'05ea6446566459559a376078d14c7ec5ef98821e4d34bab6c788ac8ebed6d952',
 'complete':'de4566acaa64208b5e41294ac9e4acc506833a23541ae65f5c0f383fa05bddcf',
 'models':'5839e78093939ff2c89c6f329ae41588023ad2121f53f8909905a59100656985',
 'groups':'801a90adf37da01108aec71acae2c9f78c5c3552248faadb61fdf2e3398c6019',
 'assign':'27a1aaff1046de55e0721f62e1cfe166c83300df7192785e02e5616c893c43c4'}

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for v in iter(lambda:f.read(1024**2),b''):h.update(v)
    return h.hexdigest()

def dose_key(v):
    if v is None or pd.isna(v) or str(v).strip()=='':raise ValueError('Dose ausente.')
    s=str(v).strip()
    try:
        x=Decimal(s)
        if not x.is_finite():raise ValueError('Dose nao finita.')
        return 'num:'+format((Decimal(0) if x==0 else x).normalize(),'f')
    except InvalidOperation:return 'txt:'+s

def check(condition,message):
    if not bool(condition):raise ValueError(message)

def close(a,b,message,rtol=1e-9,atol=1e-10):
    if not np.allclose(a,b,rtol=rtol,atol=atol,equal_nan=True):raise ValueError(message)

def contrast(x):
    x=np.asarray(x,float)
    check(x.ndim==2 and x.shape[1]==4 and np.isfinite(x).all() and (x>=0).all(),'Contraste requer quatro canais finitos nao negativos.')
    l=np.log1p(x)
    return l-l.mean(axis=1,keepdims=True)

def components(a,e):
    ca,ce=contrast(a),contrast(e)
    delta=ca-ce
    close(ca,ce+delta,'Falha da identidade C_A=C_E+Delta.')
    return ca,ce,delta

def predict_fixed(a,model,mapping):
    a=np.asarray(a,float); x=np.log1p(a)
    check(np.isfinite(x).all(),'Projecao: dados nao finitos.')
    if model['espaco']=='contrastes_4canais':x-=x.mean(1,keepdims=True)
    check(model['channels']==CHANNELS,'Ordem dos canais divergiu.')
    scale=np.asarray(model['scale']);check(np.all(scale>0),'Escala invalida no modelo salvo.')
    z=(x-np.asarray(model['center']))/scale
    sq=((z[:,None,:]-np.asarray(model['cluster_centers'])[None,:,:])**2).sum(2)
    ids=sq.argmin(1)
    return np.array([mapping[int(i)] for i in ids]),np.sqrt(sq[np.arange(len(a)),ids])

def load_inputs(root):
    import polars as pl
    paths={k:Path(root)/v for k,v in SOURCES.items()}
    hashes={}
    for k,p in paths.items():
        if not p.is_file():raise FileNotFoundError(f'Fonte fixa ausente ({k}): {p}. Nao selecionar outra execucao automaticamente.')
        hashes[k]=sha256(p)
        if k in HASHES and hashes[k]!=HASHES[k]:raise ValueError(f'Integridade divergiu: {k}. Preserve os arquivos originais e confira a fonte.')
    scores=pd.DataFrame(pl.read_parquet(paths['scores']).to_dict(as_series=False))
    frames={k:pd.DataFrame(pl.read_csv(paths[k],infer_schema_length=10000,
              schema_overrides={D:pl.String} if k in ('complete','assign') else None).to_dict(as_series=False))
              for k in ('refs','complete','groups','assign')}
    return scores,frames['refs'],frames['complete'],json.loads(paths['models'].read_text()),frames['groups'],frames['assign'],paths,hashes

def validate_inputs(scores,refs,complete,models,groups,assign):
    scores=scores.copy();complete=complete.copy();assign=assign.copy()
    required=[P,R,E,D,W]+[f'{kind}_{b}' for b in BLOCKS for kind in ['E','A']]
    check(set(required).issubset(scores),'Colunas de escores ausentes.')
    check(set(scores[R])==set(REPS),'Arquivo inclui replica fora de R1-R3. Auditoria interrompida.')
    check(len(scores)==7998,'Numero de pocos diferente de 7998.')
    check(not scores[[P,R,W]].isna().any().any() and not scores.duplicated([P,R,W]).any(),'Pocos sem chave ou duplicados.')
    check(set(scores[P])==set(PLATES),'Placas divergentes.')
    dmso=scores.loc[scores[E].eq('DMSO')].copy()
    check(len(dmso)==588 and (dmso.groupby([P,R]).size()==28).all() and len(dmso.groupby([P,R]))==21,'Esperados 28 DMSO em cada um dos 21 grupos.')
    refs=refs.loc[refs.bloco.isin(BLOCKS)].copy()
    check(len(refs)==49 and not refs.duplicated([P,'bloco']).any(),'Referencias principais incompletas/duplicadas.')
    check((refs['n_dmso_validos']==84).all() and np.isclose(refs['quantil'],.95).all(),'Referencias nao correspondem a P95 de 84 controles.')
    for b,n in EXPECTED_COUNTS.items():check((refs.loc[refs.bloco.eq(b),'n_medidas']==n).all(),f'Numero de medidas mudou: {b}')
    q=refs.pivot(index=P,columns='bloco',values='limiar_RMS').loc[PLATES,BLOCKS]
    check(np.isfinite(q).all().all() and (q>0).all().all(),'Limite P95 invalido.')
    audit=[]
    for p in PLATES:
        ctl=dmso.loc[dmso[P].eq(p)]
        for b in BLOCKS:
            ev=ctl['E_'+b].to_numpy(float);check(np.isfinite(ev).all() and (ev>=0).all(),'DMSO invalido.')
            observed=float(np.quantile(ev,.95,method='linear'));orig=float(q.loc[p,b])
            close(observed,orig,'P95 original nao reproduzido: '+p+'/'+b)
            sp=scores.loc[scores[P].eq(p)]
            close(sp['E_'+b]/orig,sp['A_'+b],'A != RMS/P95: '+p+'/'+b)
            audit.append({P:p,'bloco':b,'n_dmso':len(ev),'P95_original':orig,'P95_recalculado_apenas_conferencia':observed,
                'erro_abs_P95':abs(observed-orig),'n_dmso_acima':int((ev>orig).sum()),'pct_dmso_acima':100*np.mean(ev>orig),
                'erro_max_abs_A':float(np.nanmax(np.abs(sp['E_'+b]/orig-sp['A_'+b])))})
    scores['_dose_key']=scores[D].map(dose_key)
    for d in (complete,assign):
        for col in KEY:d[col]=d[col].astype(str)
        check(not d.duplicated(KEY).any(),'Condicoes duplicadas.')
        check(len(d)==2447,'Esperadas 2447 condicoes completas.')
    complete=complete.sort_values(KEY).reset_index(drop=True)
    assign=assign.sort_values(KEY).reset_index(drop=True)
    check(complete[KEY].equals(assign[KEY]),'As condicoes do agrupamento nao coincidem com o mapa multibloco.')
    check(not any('R4' in str(c) for c in scores.columns),'Nao se permitem colunas de R4.')
    treatment=scores.loc[~scores[E].eq('DMSO')].copy()
    check(len(treatment)==7410,'Numero de tratamientos divergiu.')
    cols=[f'{v}_{b}' for b in BLOCKS for v in ['E','A']]
    agg=treatment.groupby(KEY+[R],dropna=False)[cols].median().reset_index()
    agg=agg.merge(complete[KEY],on=KEY,how='inner',validate='many_to_one')
    check(len(agg)==3*2447,'Agregacao nao tem 3 perfis por condicao completa.')
    for r in REPS:
        ar=agg.loc[agg[R].eq(r)].sort_values(KEY)
        check(ar[KEY].reset_index(drop=True).equals(complete[KEY]),'Alinhamento de replica divergiu.')
        for b in BLOCKS:close(ar['A_'+b],complete['A_'+b+'_'+r],'Escores da tabela completa nao reproduzidos.')
    check(np.isfinite(agg[cols].to_numpy(float)).all(),'Condicoes completas contem escores invalidos.')
    frozen={}
    for space in ['magnitude_4canais','contrastes_4canais']:
        g=groups.loc[groups.espaco.eq(space)];ks=g.k.unique();check(len(ks)==1,'Mais de uma solucao de referencia.')
        model=[m for m in models if m['espaco']==space and m['k']==int(ks[0])];check(len(model)==1,'Modelo salvo ambiguo.')
        mapping=dict(zip(g.id_modelo.astype(int),g.grupo.astype(str)))
        pred,_=predict_fixed(complete[['A_'+b+'_mediana' for b in CHANNELS]],model[0],mapping)
        check(np.array_equal(pred,assign[space+'_grupo'].to_numpy()),'Modelo congelado nao reproduz atribuicoes originais.')
        frozen[space]=(model[0],mapping)
    return scores,dmso,refs,q,complete,agg,frozen,pd.DataFrame(audit)



### Resumos, contrastes e projecoes sem reajuste

In [ ]:
def control_summary(dmso,q,by_rep=True):
    rows=[];by=[P,R] if by_rep else [P]
    for k,sub in dmso.groupby(by):
        k=k if isinstance(k,tuple) else (k,)
        for b in BLOCKS:
            ev=sub['E_'+b].to_numpy(float);av=sub['A_'+b].to_numpy(float);limit=q.loc[k[0],b]
            rec=dict(zip(by,k));rec.update(bloco=b,n=len(sub),mediana_RMS=float(np.median(ev)),
                RMS_q25=float(np.quantile(ev,.25)),RMS_q75=float(np.quantile(ev,.75)),RMS_P95_diagnostico=float(np.quantile(ev,.95)),RMS_max=float(ev.max()),
                P95_original=float(limit),P95_diagnostico_sobre_original=float(np.quantile(ev,.95)/limit),
                mediana_A=float(np.median(av)),A_q25=float(np.quantile(av,.25)),A_q75=float(np.quantile(av,.75)),
                n_acima_original=int((av>1).sum()),pct_acima_original=float(100*np.mean(av>1)),cauda_P95_sobre_mediana=float(np.quantile(ev,.95)/np.median(ev)))
            rows.append(rec)
    return pd.DataFrame(rows)

def add_contrasts(frame):
    frame=frame.copy();ca,ce,delta=components(frame[['A_'+b for b in CHANNELS]],frame[['E_'+b for b in CHANNELS]])
    for j,b in enumerate(CHANNELS):
        frame['C_A_'+b]=ca[:,j];frame['C_E_'+b]=ce[:,j];frame['Delta_P95_'+b]=delta[:,j]
    return frame

def control_treatment_summaries(dmso,agg):
    cols=[prefix+b for prefix in ['C_A_','C_E_','Delta_P95_'] for b in CHANNELS]
    parts=[]
    for sample,data in [('DMSO',dmso),('tratamentos_completos',agg)]:
        for (p,r),d in data.groupby([P,R]):
            for b in CHANNELS:
                parts.append({P:p,R:r,'amostra':sample,'canal':b,'n':len(d),
                    **{pre+'media':float(d[pre+b].mean()) for pre in ['C_A_','C_E_','Delta_P95_']},
                    **{pre+'mediana':float(d[pre+b].median()) for pre in ['C_A_','C_E_','Delta_P95_']}})
    rep=pd.DataFrame(parts)
    plate=rep.groupby([P,'amostra','canal'])[['C_A_media','C_E_media','Delta_P95_media']].mean().reset_index()
    rho=[]
    for b in CHANNELS:
        for scale in ['C_A_media','C_E_media']:
            wide=plate.loc[plate.canal.eq(b)].pivot(index=P,columns='amostra',values=scale)
            val=wide.DMSO.corr(wide.tratamentos_completos,method='spearman')
            leaves=[wide.drop(p).DMSO.corr(wide.drop(p).tratamentos_completos,method='spearman') for p in PLATES]
            rho.append({'canal':b,'escala':scale,'n_placas':7,'rho_Spearman_descritivo':val,'rho_min_sem_uma_placa':min(leaves),'rho_max_sem_uma_placa':max(leaves)})
    return rep,plate,pd.DataFrame(rho)

def projections(dmso,agg,frozen,assign,complete):
    rows=[];detailed=[]
    for space,(model,mapping) in frozen.items():
        for sample,data in [('DMSO',dmso),('tratamentos_completos',agg)]:
            labels,dist=predict_fixed(data[['A_'+b for b in CHANNELS]],model,mapping)
            base=data[[P,R]+([W] if sample=='DMSO' else [E,'_dose_key'])].copy()
            base['amostra']=sample;base['espaco']=space;base['grupo']=labels;base['distancia_centro']=dist
            detailed.append(base)
            if sample=='tratamentos_completos':
                # Independent readback of all three original single-replicate projections.
                for r in REPS:
                    ar=base.loc[base[R].eq(r)].sort_values(KEY).reset_index(drop=True)
                    close_keys=ar[KEY].equals(assign.sort_values(KEY).reset_index(drop=True)[KEY])
                    check(close_keys,'Falha ao verificar projecoes de tratamentos.')
                    check(np.array_equal(ar.grupo,assign.sort_values(KEY)[space+'_'+r]),'Atribuicoes de replica nao reproduzidas.')
            for (p,r),sub in base.groupby([P,R]):
                for g in sorted(mapping.values()):
                    n=int(sub.grupo.eq(g).sum());rows.append({'espaco':space,'amostra':sample,P:p,R:r,'grupo':g,'n_grupo':n,'n_total':len(sub),'pct_grupo':100*n/len(sub)})
    return pd.DataFrame(rows),pd.concat(detailed,ignore_index=True)

def quantile_support(dmso,q):
    rows=[]
    for p in PLATES:
        for b in BLOCKS:
            z=dmso.loc[dmso[P].eq(p),[R,W,'E_'+b]].sort_values('E_'+b,kind='stable').reset_index(drop=True)
            pos=.95*(len(z)-1);lo=int(np.floor(pos));hi=int(np.ceil(pos));weight=pos-lo
            x,y=float(z.loc[lo,'E_'+b]),float(z.loc[hi,'E_'+b])
            reproduced=(1-weight)*x+weight*y
            close(reproduced,q.loc[p,b],'Interpolacao P95 nao reproduzida.')
            rows.append({P:p,'bloco':b,'n':len(z),'posicao_zero_based':pos,'indice_inferior':lo,'indice_superior':hi,
                'replica_inferior':z.loc[lo,R],'poco_inferior':z.loc[lo,W],'RMS_inferior':x,
                'replica_superior':z.loc[hi,R],'poco_superior':z.loc[hi,W],'RMS_superior':y,
                'peso_superior':weight,'P95_original':float(q.loc[p,b])})
    return pd.DataFrame(rows)


def reference_identity(q):
    # Same synthetic RMS E=1 in all four channels; ONLY a formula illustration.
    a=1/q[CHANNELS].to_numpy();c=contrast(a)
    return pd.DataFrame([{P:p,'canal':b,'RMS_ilustrativo_constante':1.,'P95_original':q.loc[p,b],
                         'A_ilustrativo':a[i,j],'C_A_ilustrativo':c[i,j]} for i,p in enumerate(PLATES) for j,b in enumerate(CHANNELS)])



### Criacao das dez figuras

In [ ]:
def save_figure(fig,out,name,caption,legends):
    fig.tight_layout()
    for ext in ('png','svg'):fig.savefig(out/'figuras'/f'{name}.{ext}',dpi=240,bbox_inches='tight')
    plt.close(fig);legends.append((name,caption))

def heatmap(data,title,label,out,name,caption,legends,vmin=None,vmax=None):
    a=data.to_numpy(float);fig,ax=plt.subplots(figsize=(max(7.5,len(data.columns)*1.5),max(4.,len(data)*.34+1.5)))
    im=ax.imshow(a,aspect='auto',vmin=vmin,vmax=vmax)
    ax.set_xticks(range(len(data.columns)),data.columns);ax.set_yticks(range(len(data)),data.index)
    ax.set_title(title,pad=15);fig.colorbar(im,ax=ax,label=label,shrink=.8)
    for i in range(len(data)):
        for j in range(len(data.columns)):
            ax.text(j,i,f'{a[i,j]:.3f}',ha='center',va='center',fontsize=8,
                    bbox=dict(boxstyle='round,pad=0.15',facecolor='white',edgecolor='none',alpha=.75))
    save_figure(fig,out,name,caption,legends)

def figures(tables,dmso,agg,q,out):
    legends=[]
    heatmap(q[CHANNELS],'Referencias P95 originais por placa','RMS normalizado',out,'D01_P95_originais',
        'P95 originalmente calculado com 84 DMSO por placa. Estes valores foram apenas conferidos e nao substituidos.',legends)
    rep=tables['DMSO_placa_replica'].query('bloco in @CHANNELS').copy();rep['linha']=rep[P]+' / '+rep[R]
    for metric,title,label,name in [('mediana_RMS','DMSO: mediana do RMS antes de dividir pelo P95','RMS ja normalizado por MAD','D02_DMSO_RMS'),
                                   ('mediana_A','DMSO: mediana do escore A apos divisao pelo P95','A = RMS / P95','D03_DMSO_A'),
                                   ('P95_diagnostico_sobre_original','DMSO: cauda de cada replica versus referencia original','P95 da replica / P95 original','D04_caudas_por_replica')]:
        heatmap(rep.pivot(index='linha',columns='bloco',values=metric)[CHANNELS],title,label,out,name,
            '28 DMSO por placa e replica. Resumos descritivos dos mesmos controles empregados na normalizacao e referencia; nao e validacao fora da amostra.',legends)
    cr=tables['contrastes_placa_replica'].query("amostra == 'DMSO'").copy();cr['linha']=cr[P]+' / '+cr[R]
    heatmap(cr.pivot(index='linha',columns='canal',values='C_A_media')[CHANNELS],
        'Contrastes dos DMSO na representacao do agrupamento','Media de C_A',out,'D05_contrastes_DMSO',
        'C_A=log(1+A) menos a media dos quatro canais no mesmo poco; sem aplicar a escala adicional do K-means.',legends)
    cp=tables['contrastes_por_placa'].query("amostra == 'DMSO' and canal == 'Mito'").set_index(P).loc[PLATES]
    fig,ax=plt.subplots(figsize=(9.5,5.2))
    for col,lab in [('C_E_media','Antes da divisao por P95: C_E'),('Delta_P95_media','Mudanca algebrica: Delta'),('C_A_media','Apos divisao: C_A')]:
        ax.plot(PLATES,cp[col],marker='o',label=lab)
    ax.axhline(0,linestyle=':',alpha=.6);ax.set_ylabel('Media do contraste Mito');ax.set_title('DMSO: C_A = C_E + Delta_P95');ax.legend(fontsize=9)
    save_figure(fig,out,'D06_decomposicao_algebrica_Mito','Identidade algebrica na escala log(1+x). Delta depende de E e de P95; nao e uma fracao causal de efeito tecnico. E ja e RMS de medidas normalizadas por MAD.',legends)
    p=tables['contrastes_por_placa'].query("canal == 'Mito'").pivot(index=P,columns='amostra',values='C_A_media')
    fig,ax=plt.subplots(figsize=(8,5.5));ax.scatter(p.DMSO,p.tratamentos_completos)
    for plate,row in p.iterrows():ax.annotate(plate,(row.DMSO,row.tratamentos_completos),xytext=(5,5),textcoords='offset points',fontsize=9)
    ax.set_xlabel('Contraste Mito medio nos DMSO');ax.set_ylabel('Contraste Mito medio nos tratamentos');ax.set_title('Correspondencia entre placas: controles e tratamentos')
    save_figure(fig,out,'D07_DMSO_vs_tratamentos','Cada ponto e uma placa, com pesos iguais para R1-R3. Tratamentos: condicoes completas agregadas por condicao e replica; controles: pocos DMSO. Ambos usam a mesma referencia P95 por placa. Associacao descritiva com sete placas, nao causal.',legends)
    pr=tables['projecoes_por_placa_replica'];sel=pr.query("espaco == 'contrastes_4canais' and amostra == 'DMSO'")
    t=sel.groupby([P,'grupo']).n_grupo.sum().unstack(fill_value=0).reindex(PLATES);pct=t.div(t.sum(1),axis=0)*100
    fig,ax=plt.subplots(figsize=(9.5,5.2));bottom=np.zeros(7)
    for g in pct:
        ax.bar(PLATES,pct[g],bottom=bottom,label=g);bottom+=pct[g].to_numpy()
    ax.set_ylim(0,100);ax.set_ylabel('% dos 84 DMSO de cada placa');ax.set_title('DMSO projetados nos grupos de contrastes ja existentes');ax.legend(title='Grupo',ncols=4)
    save_figure(fig,out,'D08_projecao_DMSO','Aplicacao do modelo congelado, sem reajuste. K-means atribui qualquer ponto a um centro. Um DMSO em G02 nao e um hit nem uma taxa de falso positivo. Distribuicao apenas diagnostica.',legends)
    z=pr.query("espaco == 'contrastes_4canais' and grupo == 'G02'").copy();z['linha']=z[P]+' / '+z[R]
    heatmap(z.pivot(index='linha',columns='amostra',values='pct_grupo')[['DMSO','tratamentos_completos']],
        'Projecao em G02: controles e tratamentos por replica','% atribuido a G02',out,'D09_G02_controles_tratamentos',
        'Mesmo modelo fixo aplicado por replica. DMSO: 28 pocos por linha; tratamentos: condicoes completas por placa. Denominadores sao distintos. Nao compara diretamente o percentual agregado das medianas de R1-R3.',legends,vmin=0,vmax=100)
    fig,ax=plt.subplots(figsize=(9.5,5.6))
    for p in PLATES:
        x=np.sort(dmso.loc[dmso[P].eq(p),'E_Mito'].to_numpy(float));ax.step(x,np.arange(1,len(x)+1)/len(x),where='post',label=p)
    ax.axhline(.95,linestyle=':',alpha=.6);ax.set_xscale('log');ax.set_xlabel('RMS Mito antes do P95 (escala log)');ax.set_ylabel('Fracao acumulada dos DMSO');ax.set_title('Distribuicoes mitocondriais dos controles');ax.legend(ncols=2,fontsize=9)
    save_figure(fig,out,'D10_DMSO_distribuicao_Mito','Distribuicoes empiricas, 84 controles por placa. A linha 0,95 ilustra o quantil original. Caudas diferentes podem coexistir com medianas proximas.',legends)
    return legends



### Relatorio e texto para o TCC

In [ ]:
def make_report(tables,checks):
    pooled=tables['DMSO_por_placa'].query("bloco == 'Mito'").set_index(P)
    cp=tables['contrastes_por_placa'].query("amostra == 'DMSO' and canal == 'Mito'").set_index(P)
    cr=tables['contrastes_placa_replica'].query("amostra == 'DMSO' and canal == 'Mito'").set_index([P,R])
    pr=tables['projecoes_por_placa_replica'];p=pr.query("espaco == 'contrastes_4canais' and amostra == 'DMSO' and grupo == 'G02'").groupby(P)[['n_grupo','n_total']].sum()
    rho=tables['correlacoes_entre_placas'].query("canal == 'Mito' and escala == 'C_A_media'").iloc[0]
    support=tables['suporte_P95_original'].query("Metadata_Plate_ID == 'B1002' and bloco == 'Mito'").iloc[0]
    tail=tables['DMSO_placa_replica'].query("Metadata_Plate_ID == 'B1002' and bloco == 'Mito'").set_index(R)
    pairs=[]
    for plate in PLATES:
        r=pooled.loc[plate];pairs.append(f'| {plate} | {r.mediana_RMS:.4f} | {r.P95_original:.4f} | {r.mediana_A:.4f} | {cp.loc[plate,"C_A_media"]:+.4f} |')
    text='\n'.join([
        '# Auditoria dos controles DMSO e referencias P95 - R1-R3','',
        'Analise descritiva pos-hoc. Nenhum modelo reajustado, escore recalibrado, candidato reclassificado ou dado de R4 utilizado.','',
        '## Conferencias',f'{checks["n_pocos"]} pocos; {checks["n_DMSO"]} DMSO; 28 DMSO por placa x replica; 49 referencias principais conferidas. As 2447 atribuicoes de cada modelo e as projecoes de R1-R3 reproduziram os arquivos originais.',
        f'Erro maximo absoluto na conferencia dos P95: {checks["erro_P95"]:.3g}. Erro maximo na identidade A=E/P95: {checks["erro_A"]:.3g}.','',
        '## Resposta a pergunta',
        'Sim: as diferencas entre placas aparecem tambem nos controles DMSO e nas referencias utilizadas. A variacao dos denominadores P95 modifica os contrastes mesmo quando as medianas dos RMS sao proximas. Isso limita a interpretacao dos grupos como exclusivamente biologicos; nao prova que toda a diferenca nos tratamentos seja tecnica.','',
        '### Mito nos controles (84 pocos por placa)',
        '| Placa | RMS mediano E | P95 original | Mediana A | Media C_A Mito |',
        '|---|---:|---:|---:|---:|',*pairs,'',
        'E e RMS de medidas JA normalizadas por MAD, nao intensidade bruta. A=E/P95. C_A=log(1+A) menos a media dos quatro canais no mesmo poco.','',
        '## Decomposicao algebrica, nao causal',
        f'B1002: C_E Mito medio={cp.loc["B1002","C_E_media"]:+.4f}; Delta_P95={cp.loc["B1002","Delta_P95_media"]:+.4f}; C_A={cp.loc["B1002","C_A_media"]:+.4f}.',
        f'B1007: C_E Mito medio={cp.loc["B1007","C_E_media"]:+.4f}; Delta_P95={cp.loc["B1007","Delta_P95_media"]:+.4f}; C_A={cp.loc["B1007","C_A_media"]:+.4f}.',
        'A identidade C_A=C_E+Delta preserva todos os dados. Delta depende simultaneamente do numerador e do denominador; nao representa porcentagem de artefato nem efeito tecnico isolado. Nenhum agrupamento foi aplicado a C_E.',
        'Em referencias_RMS_constante.csv ha uma ilustracao SEPARADA com E=1 nos quatro canais. Sao valores artificiais exclusivamente para demonstrar a formula, nao amostras observadas.','',
        '## Consistencia por replica',
        *[f'{plate}: '+', '.join(f'{r}={cr.loc[(plate,r),"C_A_media"]:+.4f}' for r in REPS)+'.' for plate in ['B1002','B1007']],
        'Os 21 grupos placa x replica nao sao tratados como 21 placas independentes.','',
        '## Cauda dos controles de B1002',
        'Excedencias do limite original por replica: '+', '.join(f'{r}: {int(tail.loc[r,"n_acima_original"])}/28' for r in REPS)+'.',
        f'O P95 mitocondrial de B1002 interpola os RMS {support.RMS_inferior:.6f} ({support.replica_inferior}/{support.poco_inferior}) e {support.RMS_superior:.6f} ({support.replica_superior}/{support.poco_superior}), com peso superior {support.peso_superior:.2f}.',
        'O maximo isolado da distribuicao nao determina sozinho o P95. Nao foi excluido nenhum controle. Referencias comuns as tres replicas podem refletir principalmente a cauda de uma delas.','',
        '## Correspondencia com tratamentos',
        f'Correlacao de Spearman descritiva entre medias dos contrastes Mito de controles e tratamentos nas sete placas: {rho.rho_Spearman_descritivo:.4f}. Retirando uma placa por vez: {rho.rho_min_sem_uma_placa:.4f} a {rho.rho_max_sem_uma_placa:.4f}.',
        'Cada placa recebe peso igual, e suas tres replicas sao resumidas com peso igual. A referencia P95 e compartilhada entre controles e tratamentos; parte da correspondencia pode ser matematica. Nao foi realizado teste de significancia nem estimada causalidade.','',
        '## Projecao dos controles no modelo congelado',
        *[f'{plate}: {int(p.loc[plate,"n_grupo"])}/{int(p.loc[plate,"n_total"])} DMSO em G02 de contrastes ({100*p.loc[plate,"n_grupo"]/p.loc[plate,"n_total"]:.2f}%).' for plate in ['B1002','B1007']],
        'Estas sao projecoes de pocos DMSO, nao probabilidades, hits ou falsos positivos. O modelo sempre escolhe um centro. A tabela por replica compara controles e tratamentos na mesma etapa de agregacao; a atribuicao original nas medianas de R1-R3 permanece intacta.','',
        '## Limites e decisao',
        'Os mesmos controles participaram da normalizacao MAD e da estimativa de P95. A taxa de excedencia no conjunto reunido e, portanto, uma verificacao da calibracao de entrada, nao uma validacao independente. Diferencas de referencia nao sao automaticamente erros: podem representar caudas mais largas de distribuicoes diferentes.',
        'Esta auditoria nao inspeciona imagens, segmentacao ou perfis anteriores a MAD; nao identifica a causa das caudas. Nao separa composicao dos compostos de todos os efeitos tecnicos nos tratamentos.',
        'Preservar os 12 candidatos, a avaliacao de R4 e o agrupamento original. Descrever a influencia das referencias sobre os contrastes e evitar interpretar os grupos como classes biologicas exclusivas. Uma eventual mudanca futura de representacao sera uma analise nova, nao uma correcao retrospectiva desta execucao.'])
    doc='\n\n'.join([
        'Auditoria dos controles DMSO e das referencias dos escores',
        'Foi realizada uma auditoria descritiva pos-hoc dos 588 controles DMSO de R1-R3, distribuidos em sete placas e tres replicas, com 28 controles por combinacao. Os RMS por bloco e os escores A foram lidos dos arquivos originais. A identidade A=RMS/P95 e os 49 limites dos sete blocos principais foram conferidos. Foram examinados mediana, quartis, cauda P95 e excedencia por placa e replica, sem alterar os limites originais.',
        'Para os quatro canais utilizados no agrupamento, compararam-se os contrastes C_A, obtidos pela centralizacao por linha de log(1+A), aos contrastes C_E, obtidos analogamente a partir do RMS anterior a divisao pelo P95. A diferenca Delta=C_A-C_E foi apresentada apenas como identidade algebrica, nao como efeito causal. As medias foram utilizadas nessa decomposicao para preservar sua aditividade. Os DMSO foram projetados nos centros e nas escalas previamente salvos, sem reajuste dos modelos; todas as atribuicoes originais dos tratamentos foram reproduzidas antes da projecao.',
        f'Os controles apresentaram diferencas entre placas tambem na representacao de contrastes. Em B1002, o RMS mitocondrial mediano foi {pooled.loc["B1002","mediana_RMS"]:.4f}, o P95 original foi {pooled.loc["B1002","P95_original"]:.4f} e a mediana de A foi {pooled.loc["B1002","mediana_A"]:.4f}. Em B1007, os valores correspondentes foram {pooled.loc["B1007","mediana_RMS"]:.4f}, {pooled.loc["B1007","P95_original"]:.4f} e {pooled.loc["B1007","mediana_A"]:.4f}. Portanto, RMS medianos relativamente proximos resultaram em escores A distintos em funcao de suas referencias de cauda.',
        f'Nos controles, o contraste mitocondrial medio apos a divisao pelo P95 foi {cp.loc["B1002","C_A_media"]:.4f} em B1002 e {cp.loc["B1007","C_A_media"]:.4f} em B1007. A projecao no modelo de contrastes previamente ajustado atribuiu G02 a {int(p.loc["B1002","n_grupo"])}/84 DMSO de B1002 e a {int(p.loc["B1007","n_grupo"])}/84 de B1007. Essa atribuicao e estritamente diagnostica: um modelo K-means atribui pontos a centros mesmo sem uma alteracao biologica demonstrada. A presenca de controles em G02 nao corresponde a uma taxa de falsos positivos.',
        'As diferencas nos controles e nas referencias mostram que a organizacao por contrastes nao pode ser atribuida exclusivamente a resposta dos compostos. Contudo, nao quantificam uma fracao tecnica nem invalidam automaticamente os candidatos originais. Os controles usados na auditoria tambem participaram da normalizacao e da estimativa das referencias; assim, o resultado nao constitui validacao independente. Nenhum dado de R4 foi utilizado nesta etapa.'])
    return text,doc



### Execucao e preservacao dos arquivos

In [ ]:
def audit_frames(scores,refs,complete,models,groups,assign,out,provenance=None):
    out=Path(out);out.mkdir(parents=True,exist_ok=False);(out/'tabelas').mkdir();(out/'figuras').mkdir()
    plan={'pergunta':'Diferencas de contraste entre placas tambem ocorrem nos DMSO de R1-R3 e referencias P95?',
        'natureza':'Auditoria descritiva pos-hoc','novos_modelos':False,'recalibracao':False,'R4_lida':False,
        'limiares_originais_preservados':True,'n_pocos_esperado':7998,'n_DMSO_esperado':588,
        'estatistica':'Resumos por placa/replica, identidades algebricas, correlacoes descritivas entre sete placas, projecoes em modelos fixos',
        'fontes':provenance or {},'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
    (out/'plano_auditoria.json').write_text(json.dumps(plan,ensure_ascii=False,indent=2))
    s,dmso,ref,q,c,agg,frozen,audit=validate_inputs(scores,refs,complete,models,groups,assign)
    dmso=add_contrasts(dmso);agg=add_contrasts(agg)
    rep,plate,rho=control_treatment_summaries(dmso,agg)
    projections_table,projection_details=projections(dmso,agg,frozen,assign,c)
    tables={'conferencia_referencias':audit,'DMSO_placa_replica':control_summary(dmso,q),
        'DMSO_por_placa':control_summary(dmso,q,False),'contrastes_placa_replica':rep,
        'contrastes_por_placa':plate,'correlacoes_entre_placas':rho,
        'projecoes_por_placa_replica':projections_table,'projecoes_diagnosticas':projection_details,
        'referencias_RMS_constante':reference_identity(q),'suporte_P95_original':quantile_support(dmso,q),
        'DMSO_escores_e_contrastes':dmso[[P,R,W]+[v+b for v in ('E_','A_','C_E_','C_A_','Delta_P95_') for b in CHANNELS]]}
    # All rankings are descriptive, never used to remove controls or change references.
    tails=[]
    for p in PLATES:
        for b in CHANNELS:
            t=dmso.loc[dmso[P].eq(p),[P,R,W,'E_'+b,'A_'+b]].sort_values('E_'+b,ascending=False).head(8).copy()
            t=t.rename(columns={'E_'+b:'RMS','A_'+b:'A'});t['canal']=b;t['ordem_RMS']=np.arange(1,len(t)+1);tails.append(t)
    tables['DMSO_maiores_RMS_inspecao']=pd.concat(tails,ignore_index=True)
    for name,table in tables.items():table.to_csv(out/'tabelas'/f'{name}.csv',index=False,float_format='%.17g')
    checks={'n_pocos':len(s),'n_DMSO':len(dmso),'n_tratamentos':len(s)-len(dmso),'n_condicoes_completas':len(c),
            'n_referencias':len(audit),'erro_P95':float(audit.erro_abs_P95.max()),'erro_A':float(audit.erro_max_abs_A.max()),
            'atribuicoes_e_projecoes_originais_reproduzidas':True}
    report,text=make_report(tables,checks);legends=figures(tables,dmso,agg,q,out)
    (out/'relatorio_auditoria_DMSO.md').write_text(report,encoding='utf8')
    (out/'texto_TCC_auditoria_DMSO.md').write_text(text,encoding='utf8')
    (out/'legendas.md').write_text('\n\n'.join(f'## {name}\n{caption}' for name,caption in legends),encoding='utf8')
    (out/'fontes.md').write_text('\n'.join(f'{k}: https://drive.google.com/file/d/{fid}/view' for k,fid in SOURCE_IDS.items()),encoding='utf8')
    manifest={**plan,'conferencias':checks,'n_figuras':len(legends),'arquivos':{str(f.relative_to(out)):sha256(f) for f in out.rglob('*') if f.is_file()}}
    (out/'manifesto_auditoria.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf8')
    return tables,report,text,manifest

def run(root):
    scores,refs,complete,models,groups,assign,paths,hashes=load_inputs(root)
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    out=Path(root)/'resultados_tcc/auditoria_DMSO_P95'/('R1_R3_'+stamp)
    provenance={k:{'path':str(paths[k]),'sha256':hashes[k],'drive_id':SOURCE_IDS[k]} for k in paths}
    tables,report,text,manifest=audit_frames(scores,refs,complete,models,groups,assign,out,provenance)
    for k,p in paths.items():check(sha256(p)==hashes[k],'Um arquivo de entrada foi modificado durante a auditoria.')
    archive=shutil.make_archive(str(out),'zip',root_dir=out)
    return out,archive,tables,report,manifest


## 3. Executar a auditoria
As verificacoes exigem 7.998 pocos, 588 DMSO e as 2.447 condicoes completas. Os 49 P95 principais sao recalculados somente para conferencia; os limites originais sao preservados. Os modelos fixos devem reproduzir todas as atribuicoes da Etapa 04.

A auditoria termina com erro explicativo se alguma fonte foi modificada. Nao desative o teste de integridade para contornar uma divergencia.

In [ ]:
OUT, PACOTE, TABELAS, RELATORIO, MANIFESTO = run(RAIZ)
print('Auditoria concluida:', OUT)
print('Pacote:', PACOTE)
display(TABELAS['conferencia_referencias'].head())


## 4. Relatorio e numeros principais
As correlacoes entre controles e tratamentos usam sete placas. As replicas nao sao contadas como placas independentes. O denominador P95 compartilhado pode contribuir matematicamente para a correspondencia.

In [ ]:
display(Markdown(RELATORIO))
display(TABELAS['DMSO_por_placa'].query("bloco == 'Mito'"))
display(TABELAS['contrastes_placa_replica'].query("amostra == 'DMSO' and canal == 'Mito'"))
display(TABELAS['suporte_P95_original'].query("bloco == 'Mito'"))


## 5. Figuras
Dez figuras em PNG e SVG. D01-D05 descrevem referencias e controles. D06 decompoe os contrastes algebricamente. D07 compara controles e tratamentos entre placas. D08-D09 projetam controles nos modelos fixos. D10 mostra as distribuicoes empiricas de RMS mitocondrial.

Os valores artificiais com RMS=1, exportados em `referencias_RMS_constante.csv`, sao apenas uma ilustracao da formula, separados das observacoes e nao utilizados nos modelos.

In [ ]:
for figure in sorted((OUT/'figuras').glob('*.png')):
    display(Markdown('### '+figure.stem))
    display(Image(filename=str(figure), width=1000))
print((OUT/'legendas.md').read_text())


## 6. Texto, arquivos e continuidade
O arquivo `texto_TCC_auditoria_DMSO.md` contem a descricao para revisao antes da inclusao no documento mestre. Esta etapa nao autoriza remover placas, corrigir escores ou promover novos candidatos. Mudancas metodologicas futuras devem constituir analises separadas.

A investigacao nao le as imagens nem os perfis anteriores a MAD, portanto nao determina a origem das caudas. A conclusao deve distinguir a influencia matematica das referencias de uma causa tecnica comprovada.

In [ ]:
display(Markdown((OUT/'texto_TCC_auditoria_DMSO.md').read_text()))
print('CONCLUIDO:', MANIFESTO['n_figuras'], 'figuras; tabelas; relatorio e texto exportados.')
print('Pasta:', OUT)
print('Pacote:', PACOTE)
print('R4 nao utilizada. Nenhum modelo reajustado. Nenhum limiar ou candidato alterado.')
